# Plant leaf disease classifier
Fine-tunes MobileNetV2 on PlantVillage (38 classes) and exports `plant_disease_model.h5` + `class_names.json`.

Set Runtime > Change runtime type > GPU first.

In [ ]:
import glob, json
import numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

## 1. Download PlantVillage from Kaggle
Upload your `kaggle.json` when prompted.

In [ ]:
from google.colab import files
files.upload()  # choose kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content --unzip -q

## 2. Load data
Images are scaled to 0-1 here, exactly as `app.py` does at prediction time.

In [ ]:
train_dir = sorted(glob.glob('/content/**/train', recursive=True))[0]
valid_dir = sorted(glob.glob('/content/**/valid', recursive=True))[0]
IMG, BS = 224, 32
train_ds = keras.utils.image_dataset_from_directory(train_dir, image_size=(IMG, IMG), batch_size=BS, label_mode='categorical')
val_ds = keras.utils.image_dataset_from_directory(valid_dir, image_size=(IMG, IMG), batch_size=BS, label_mode='categorical', shuffle=False)
class_names = train_ds.class_names
print(len(class_names), 'classes')
scale = lambda x, y: (x / 255.0, y)
train_ds = train_ds.map(scale).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(scale).prefetch(tf.data.AUTOTUNE)

## 3. Build the CNN

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = keras.Input((IMG, IMG, 3))
x = layers.RandomFlip('horizontal')(inputs)        # augmentation: active only while training
x = layers.RandomRotation(0.1)(x)
x = layers.Rescaling(2.0, offset=-1.0)(x)          # 0-1 -> -1..1, what MobileNetV2 expects
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

## 4. Train: head first, then fine-tune the top layers

In [ ]:
h1 = model.fit(train_ds, validation_data=val_ds, epochs=3)

base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=3)

In [ ]:
import matplotlib.pyplot as plt
acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
plt.plot(acc, marker='o'); plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.show()
print('Final validation accuracy: %.2f%%' % (model.evaluate(val_ds, verbose=0)[1] * 100))

## 5. Export
Copy both downloaded files into the project's `model/` folder.

In [ ]:
model.save('plant_disease_model.h5')
json.dump(class_names, open('class_names.json', 'w'))
files.download('plant_disease_model.h5'); files.download('class_names.json')